## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [84]:
import numpy as np
import pandas as pd
import seaborn as sns

titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
633,0,1,male,NaN,0,0,0.0000,S,First,man,True,NaN,Southampton,no,True
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
218,1,1,female,32.0,0,0,76.2917,C,First,woman,False,D,Cherbourg,yes,True
610,0,3,female,39.0,1,5,31.2750,S,Third,woman,False,NaN,Southampton,no,False
860,0,3,male,41.0,2,0,14.1083,S,Third,man,True,NaN,Southampton,no,False


**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [85]:
def get_miss_data_count(data_frame: pd.DataFrame) -> pd.Series:
    return data_frame.isnull().sum()

get_miss_data_count(titanic_data)

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [86]:
def drop_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(thresh=len(df) // 2, axis='columns')
    df = df.dropna(thresh=len(df.columns) // 2, axis='rows')
    return df

titanic_data = drop_columns(titanic_data)

### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [87]:
def fill_age_column(titanic_data: pd.DataFrame) -> pd.DataFrame:
    for who in ['man', 'woman', 'child']:
        mask = (titanic_data['who'] == who) & (titanic_data['age'].isna())
        median_age = round(titanic_data.loc[titanic_data['who'] == who, 'age'].median())
        titanic_data.loc[mask, 'age'] = median_age
    
    return titanic_data

titanic_data = fill_age_column(titanic_data)

### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [88]:
def drop_rows(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(axis='rows')
    return df

titanic_data = drop_rows(titanic_data)

### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [89]:
def get_most_common_town(titanic_data: pd.DataFrame) -> str:
    return titanic_data['embark_town'].mode()[0]

print(f"Most common town: {get_most_common_town(titanic_data)}")

Most common town: Southampton


### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [ ]:
def get_survival_rate(titanic_data: pd.DataFrame) -> float:
    return round((titanic_data['alive'] == 'yes').mean() * 100, 2)

print(f"Survived {get_survival_rate(titanic_data)}% of passangers")

Survived 38.25% of passagers


### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [91]:
def get_survive_town_statistic(titanic_data: pd.DataFrame) -> pd.Series:
    towns = titanic_data['embark_town'].unique()
    index = pd.Index(towns)
    statistic = pd.Series(index=index, data=[0] * index.size)

    alive_mask = titanic_data['alive'] == 'yes'
    for town in towns:
        town_mask = titanic_data['embark_town'] == town
        mask = alive_mask & town_mask
        statistic.loc[town] = mask.sum()

    return statistic

print(get_survive_town_statistic(titanic_data))

Southampton    217
Cherbourg       93
Queenstown      30
dtype: int64


### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [92]:
def get_survive_class_statistic(titanic_data: pd.DataFrame) -> pd.Series:
    classes = titanic_data['class'].unique()
    index = pd.Index(classes)
    statistic = pd.Series(index=index, data=[0] * index.size, dtype=np.float64)


    alive_mask = titanic_data['alive'] == 'yes'
    for cls in classes:
        cls_mask = titanic_data['class'] == cls
        mask = alive_mask & cls_mask
        statistic.loc[cls] = round(mask.sum() / cls_mask.size * 100, 2)

    return statistic

print(get_survive_class_statistic(titanic_data))

Third     13.39
First     15.07
Second     9.79
dtype: float64


### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [ ]:
def get_rich_survival_rate(titanic_data: pd.DataFrame) -> float:
    rich_passagers = titanic_data[titanic_data['fare'] >= 100]

    rich_survive_count = (rich_passagers['alive'] == 'yes').sum()
    rich_passagers_total = len(rich_passagers)

    rate = rich_survive_count / rich_passagers_total * 100

    return round(rate, 2)

print(f'Rich passengers survival rate: {get_rich_survival_rate(titanic_data)}')

Rich passagers survival percent: 73.58


### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [95]:
def get_alone_children_count(titanic_data: pd.DataFrame) -> int:
    children_alone = len(titanic_data[(titanic_data['who'] == 'child') & (titanic_data['alone'])])
    return children_alone

print(f'Children who traveled alone: {get_alone_children_count(titanic_data)}')

Children who traveled alone: 6


Какие выводы вы можете сделать о выживших пассажирах Титаника? 